# Laptop lab 1: Visualization
*Look at your data before you model it*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, October 29, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab01.ipynb)

This lab practices Lecture 2 in the same order: why we plot, what a plot reveals, what makes a good plot, how to plot in Python, and how to check the data before any plot.

> **First:** choose **File → Save a copy in Drive** and work in your copy. Colab does not keep changes to the original.

## What you should be able to do afterwards

- Load a table with pandas and answer questions about it.
- Make a line plot, a scatter plot, a histogram, a bar chart and a heatmap with matplotlib.
- Choose a chart and a colormap that fit the question.
- Check a dataset for problems before you plot or model it.

## How this lab works

| You see | What you do |
|---|---|
| **Example** | Run the cell and read the code. It shows the idea. |
| **Your turn** | Write code in the cell: replace the `...`. |
| **Check** | Run the check cell below your code. It says ✓, or tells you what to look at. |
| **Hint** | Click to unfold, when you are stuck. |
| **Explore** | Move the slider or pick from the menu, and watch the plot change. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss some of them in the lab. |

Run the cells from top to bottom with **Shift + Enter**. Stuck for more than five minutes? Raise your hand: that is what the lab is for.

| Part | Lecture 2 | Minutes |
|---|---|---|
| 0. Setup | | 5 |
| 1. Python in ten minutes (optional) | | 10 |
| 2. The course data: a student survey | The course data | 15 |
| 3. Why visualize? | Anscombe's quartet | 10 |
| 4. What a plot reveals | Correlation, data types | 10 |
| 5. Principles of good plots | Channels, colormaps | 10 |
| 6. Plotting in Python | Line, scatter, histogram, bar, heatmap | 25 |
| 7. Checking data quality | Four lines before any plot | 20 |
| 8. Challenge: Dutch provinces | | if time is left, or at home |
| Check your understanding | Exam-style questions | 5 |

## Part 0. Setup

Run the cell below. It loads the libraries, the data and the checks of this lab. It takes a few seconds.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# The data and the checks of this lab are in the course lab repository.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab01_tools.py", "lab01_tools.py")
from lab01_tools import *

SURVEY = BASE + "data/student_survey.csv"
SURVEY_RAW = BASE + "data/student_survey_raw.csv"
ANSCOMBE = BASE + "data/anscombe.csv"
TRENDS = BASE + "data/phds_arcades.csv"
PROVINCES = BASE + "data/cbs_provinces.csv"

setup_ok()

A green box means you are ready. A red error message? Run the cell again. If it stays red, raise your hand.

Colab may warn that the notebook was not authored by Google: click **Run anyway** (see the Laptop lab guide).

## Part 1. Python in ten minutes (optional)

Do you write Python regularly? Skip to Part 2.

Four ideas are enough for today. A **variable** stores a value. A **list** stores several values. A **`for` loop** repeats a step for each value. A **function** packs steps under a name.

**Example.** Run the cell and read it line by line.

In [ ]:
hours = [12.5, 8.0, 20.0, 3.5]     # a list of study hours per week

total = 0
for h in hours:                     # repeat for each value in the list
    total = total + h               # the indented lines belong to the loop
print("total:", total)
print("mean:", total / len(hours))  # len() gives the length of a list

def per_day(hours_per_week):        # a function: from input to output
    return hours_per_week / 7

print("14 hours per week is", per_day(14), "hours per day")

**Predict, then run.** What does the next cell print? Decide first, then run it.

In [ ]:
x = [3, 1, 2]
x.append(5)          # add a value at the end
print(len(x), x[0], x[-1])

<details><summary><b>Answer</b> (click to unfold)</summary>

It prints `4 3 5`. After `append`, the list is `[3, 1, 2, 5]`: four values. Python counts from 0, so `x[0]` is the first value; `x[-1]` is the last one.

</details>

**Your turn 1.1.** The pass mark of this course is 5.5. Write a function `passed(grade)` that returns `True` when the grade is at least 5.5, and `False` otherwise.

In [ ]:
def passed(grade):
    ...

In [ ]:
# Check
check_1_1()

<details><summary><b>Hint</b> (click to unfold)</summary>

A comparison such as `grade >= 5.5` is already `True` or `False`. A function hands back its result with `return`.

</details>

**Your turn 1.2.** Use a loop and your function to count how many of the grades below pass. Store the count in `n_passed`.

In [ ]:
grades = [6.1, 4.8, 5.5, 7.3, 3.9, 9.0]
n_passed = 0
for g in grades:
    ...

In [ ]:
# Check
check_1_2()

<details><summary><b>Hint</b> (click to unfold)</summary>

Inside the loop: `if passed(g):` and, indented below it, add 1 to `n_passed`.

</details>

**Why pandas?** A loop is fine for six grades. For a table of thousands of rows we use **pandas**: it does the same work on a whole column at once.

In [ ]:
grades = pd.Series([6.1, 4.8, 5.5, 7.3, 3.9, 9.0])
(grades >= 5.5).sum()     # True counts as 1, False as 0

## Part 2. The course data: a student survey

We use the survey of Lecture 2: 300 synthetic students in an introductory course. The data are generated for the course; they describe no real students.

| Column | Meaning |
|---|---|
| `program` | Economics, Business, Communication, Psychology or Other |
| `age` | age in years |
| `study_hours` | hours of study **per week** |
| `social_media_hours` | hours of social media **per day** |
| `sleep_hours` | hours of sleep **per night** |
| `satisfaction` | satisfaction with the course, 1 to 5 |
| `grade` | final grade, 1 to 10 |
| `passed` | `True` when the grade is at least 5.5 |

In the words of Lecture 1: one row is one **example** (a student), and the columns are **features**. If we predict `passed`, the label is a category: **classification**. If we predict `grade`, the label is a number: **regression**.

**Example.** Load the table and show the first five rows.

In [ ]:
df = pd.read_csv(SURVEY)
df.head()

Three ways to look at a table, before anything else:

```python
df.shape        # (rows, columns)
df.dtypes       # the data type of each column
df.describe()   # count, mean, min, max, ... of each numeric column
```

**Your turn 2.1.** Run the three lines, each in its own cell or one after the other. Then store the number of rows and columns in `n_rows` and `n_cols`.

In [ ]:
# Try df.shape, df.dtypes and df.describe() here.

n_rows = ...
n_cols = ...

In [ ]:
# Check
check_2_1()

<details><summary><b>Hint</b> (click to unfold)</summary>

`df.shape` is a pair. `n_rows, n_cols = df.shape` unpacks it into two variables at once.

</details>

**Think.** `df.describe()` leaves out some columns. Which ones, and why?

*Your answer:*

**Selecting columns and rows.** Square brackets select:

```python
df["grade"]                    # one column (a Series)
df[["grade", "passed"]]        # a table with two columns
df[df["study_hours"] > 20]     # the rows where the condition is True
```

**Your turn 2.2.** How many students study more than 20 hours per week? Store the number in `n_hard_workers`. What share of *them* passed? Store it, as a number between 0 and 1, in `share_passed`.

In [ ]:
hard_workers = ...
n_hard_workers = ...
share_passed = ...

In [ ]:
# Check
check_2_2()

<details><summary><b>Hint</b> (click to unfold)</summary>

`len(table)` counts rows. The mean of a True/False column is the share of `True`: `hard_workers["passed"].mean()`.

</details>

**Counting and grouping.** `value_counts()` counts the values of a column. `groupby` splits the table into groups and summarizes each group:

In [ ]:
print(df["program"].value_counts())
df.groupby("program")["grade"].mean()

**Your turn 2.3.** Compute the mean grade per program, sorted from low to high. Store it in `means`: you need it for the bar chart in Part 6.

In [ ]:
means = ...
means

In [ ]:
# Check
check_2_3()

<details><summary><b>Hint</b> (click to unfold)</summary>

Add `.sort_values()` at the end of the `groupby` line above.

</details>

**New columns.** A computation on columns gives a new column:

```python
df["study_hours_per_day"] = df["study_hours"] / 7
```

**Your turn 2.4.** How much social media per hour of study? Careful with the units: study hours are per **week**, social media hours per **day**.

1. Add a column `social_media_week`: social media hours per week.
2. Add a column `screen_ratio`: `social_media_week` divided by `study_hours`.
3. Which program has the highest mean `screen_ratio`? Store its name in `top_program`.

In [ ]:
df["social_media_week"] = ...
df["screen_ratio"] = ...
top_program = ...

In [ ]:
# Check
check_2_4()

<details><summary><b>Hint</b> (click to unfold)</summary>

For step 3: group by program and take the mean of `screen_ratio`, as in 2.3. `.idxmax()` gives the label of the largest value.

</details>

## Part 3. Why visualize?

Lecture 2 gives two reasons to make a plot. **Explore:** for yourself, to find patterns, errors and surprises. **Communicate:** for others, to show a finding clearly and honestly. In machine learning, exploring comes first: plots tell you which problems to fix and which models to try.

Anscombe's quartet shows why. It is four small datasets, I to IV, with 11 points each.

**Example.** Load the quartet and compare the four datasets by their summary numbers.

In [ ]:
ans = pd.read_csv(ANSCOMBE)
print(ans.head())

stats = ans.groupby("dataset").agg(mean_x=("x", "mean"), mean_y=("y", "mean"),
                                   var_x=("x", "var"), var_y=("y", "var"))
stats["corr"] = [ans[ans["dataset"] == d]["x"].corr(ans[ans["dataset"] == d]["y"]) for d in stats.index]
stats.round(2)

**Think.** From these numbers only: are the four datasets alike?

*Your answer:*

**Your turn 3.1.** Draw the four datasets side by side, one scatter plot each. The loop is given; add the scatter plot.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharex=True, sharey=True)
for ax, name in zip(axes, ["I", "II", "III", "IV"]):
    part = ans[ans["dataset"] == name]
    ...                                  # draw part["x"] against part["y"]
    ax.set_title(f"Dataset {name}")
plt.show()

In [ ]:
# Check
check_3_1()

<details><summary><b>Hint</b> (click to unfold)</summary>

`ax.scatter(x, y)` draws points on the axes `ax`. Here `x` is `part["x"]` and `y` is `part["y"]`.

</details>

**Think.** Describe each dataset in a few words.

*Your answer:*

> **Lesson.** Summary statistics can hide the shape of the data. Always plot it (Anscombe, 1973).

## Part 4. What a plot reveals

### Correlation is not causation

Lecture 2 showed two series from the United States, 2000 to 2009: the number of math PhDs awarded, and the revenue of arcades (in million USD).

**Your turn 4.1.** Load the data and compute the correlation between the two columns. Store it in `r`.

In [ ]:
trends = pd.read_csv(TRENDS)
print(trends)

r = ...

In [ ]:
# Check
check_4_1()

<details><summary><b>Hint</b> (click to unfold)</summary>

`series_a.corr(series_b)` gives the correlation of two columns.

</details>

**Explore.** The plot below draws the two series on **twin axes**: each has its own y-axis. Move the slider to change the scale of the right axis. Can you make the two lines lie on top of each other? Can you make them look unrelated?

In [ ]:
explore_twin_axes(trends)

**Think.** Does arcade revenue produce mathematicians? Why are the two series so strongly correlated?

*Your answer:*

### Wrong data type

Text is sorted alphabetically: Apr, Aug, Dec, ... This happens, for example, when a month is stored as text and the table is sorted, or grouped, by it.

**Example.** Course sign-ups per month, sorted as text.

In [ ]:
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
signups = pd.Series([52, 66, 97, 128, 149, 163, 155, 126, 104, 73, 51, 38], index=months)
wrong = signups.sort_index()          # sorts the month names alphabetically

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(wrong.index, wrong.values, marker="o")
ax.set_ylabel("course sign-ups")
plt.show()

**Your turn 4.2.** Draw the line again, with the months in calendar order. `wrong.reindex(months)` puts the rows in the order of the list `months`.

In [ ]:
right = ...

fig, ax = plt.subplots(figsize=(6, 3))
...
ax.set_ylabel("course sign-ups")
plt.show()

In [ ]:
# Check
check_4_2()

> **Check.** Is every column stored as what it is: a number, a date, a category?

Two more things a plot reveals, a **definition mismatch** and a **typing error**, are hidden in the raw survey. You find them in Part 7.

## Part 5. Principles of good plots

Two principles from Lecture 2 (Tufte, 1983): **above all else, show the data**, and **maximize the data-ink ratio**: remove ink that does not depend on the data, such as 3D effects, heavy grids and shadows.

### Which bar is the largest?

**Explore.** Here are five numbers as a pie. Which slice is the largest? Pick your answer and click **Check**.

In [ ]:
pie_puzzle()
quiz_pie()

**Your turn 5.1.** Draw the same five numbers as bars.

In [ ]:
labels = ["A", "B", "C", "D", "E"]
values = [21, 19, 20, 22, 18]

fig, ax = plt.subplots(figsize=(4, 3))
...
plt.show()

In [ ]:
# Check
check_5_1()

> **Visual channels, from most to least accurate:** position > length > angle > area > color (after Cleveland and McGill, 1984). Use color for categories, not for amounts.

### Three kinds of colormaps

- **Sequential:** amounts, for example population density (`viridis`).
- **Diverging:** deviations from a midpoint, for example profit and loss, or correlations (`RdBu`).
- **Qualitative:** groups, for example study programs (`tab10`).

**Explore.** The same smooth hill in the colormap of your choice, and the same image printed in gray. Start with `jet`, the rainbow. Do you see rings? Are they in the data? Then try `viridis`.

In [ ]:
explore_colormaps()

**Think.** What goes wrong with `jet`?

*Your answer:*

**Explore.** Which kind of colormap fits? Click your answers.

In [ ]:
quiz_colormaps()

### Which chart?

Lecture 2 ended its first session with this question: pick a line plot, a scatter plot, a histogram or a bar chart. In Part 6 you make each of them.

In [ ]:
quiz_which_chart()

## Part 6. Plotting in Python

| Library | |
|---|---|
| `matplotlib` | the foundation; full control |
| `pandas` | quick plots of a table: `df.plot()` |
| `seaborn` | statistical plots in one line |
| `plotly`, `bokeh` | interactive plots in the browser (Part 7) |

Learn `matplotlib` first: the others build on it or copy its ideas.

### Figures and axes

A **figure** is the whole image. An **axes** is one drawing area in it, with its own x- and y-axis. Every plot command is a method of an axes: `ax.plot`, `ax.set_title`, and so on. As in the lecture, we always start with `fig, ax = plt.subplots()`.

**Example.** One figure, two axes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 3))     # 1 row, 2 columns
axes[0].plot([0, 1, 2, 3], [1, 3, 2, 4])
axes[0].set_title("axes[0]: a line")
axes[1].scatter([1, 2, 3, 4], [2, 1, 4, 3])
axes[1].set_title("axes[1]: points")
plt.show()

Every plot in this part follows the same three steps:

```python
fig, ax = plt.subplots(figsize=(6, 4))          # 1. a figure with one axes
ax.scatter(df["study_hours"], df["grade"])      # 2. draw on the axes
ax.set_xlabel("study hours per week")           # 3. label it
ax.set_ylabel("grade")
```

### Line plot: change over time

**Your turn 6.1.** The study hours per week of two programs during the seven weeks of the course. Draw one line per program, with markers, a label for each line, and axis labels. Then add a legend.

In [ ]:
weeks = [1, 2, 3, 4, 5, 6, 7]
econ = [9, 10, 10, 11, 12, 14, 18]
psych = [8, 8, 9, 9, 11, 13, 17]

fig, ax = plt.subplots()
...
plt.show()

In [ ]:
# Check
check_6_1()

<details><summary><b>Hint</b> (click to unfold)</summary>

`ax.plot(weeks, econ, marker="o", label="Economics")`, then the same for `psych`. `ax.legend()` shows the labels.

</details>

A line connects points in order. Use it only when the order of the x-axis means something, such as time.

### Scatter plot: two numbers

**Your turn 6.2.** Do students who study more get higher grades? Draw `study_hours` against `grade`, with transparent dots (`alpha=0.6`) and axis labels.

In [ ]:
fig, ax = plt.subplots()
...
plt.show()

In [ ]:
# Check
check_6_2()

**Explore.** What does `alpha` do? Move the sliders. With `alpha=1`, overlapping dots hide each other; with a small `alpha`, crowded areas show as darker.

In [ ]:
explore_alpha(df)

### Histogram: the distribution of one number

**Your turn 6.3.** How long do the students sleep? Draw a histogram of `sleep_hours` with 20 bins, and label the axes.

In [ ]:
fig, ax = plt.subplots()
...
plt.show()

In [ ]:
# Check
check_6_3()

**Explore.** Lecture 2 said: try `bins=5` and `bins=100`. Here is a slider.

In [ ]:
explore_bins(df["sleep_hours"], xlabel="sleep hours per night")

**Think.** What happens with very few bins, and with very many?

*Your answer:*

### Bar chart: one number per group

**Your turn 6.4.** Draw the mean grade per program as **horizontal** bars: `ax.barh`. Use `means` from Exercise 2.3, which is already sorted.

In [ ]:
fig, ax = plt.subplots()
...
plt.show()

In [ ]:
# Check
check_6_4()

Horizontal bars leave room for long labels; sorting makes the ranking visible.

**Think.** The five means lie between 4.6 and 5.2. What does a bar of means hide? Run the example below and compare.

In [ ]:
fig, ax = plt.subplots()
order = df.groupby("program")["grade"].mean().sort_values().index
sns.boxplot(data=df, x="grade", y="program", order=order, ax=ax)
plt.show()

*Your answer:*

### Heatmap: a table of numbers

**Your turn 6.5.** Draw the correlations between four columns as a heatmap. A correlation lies between -1 and 1 with a natural midpoint at 0, so choose a **diverging** colormap and fix the scale with `vmin=-1, vmax=1`. Add a colorbar.

In [ ]:
cols = ["study_hours", "social_media_hours", "sleep_hours", "grade"]
corr = df[cols].corr()

fig, ax = plt.subplots()
im = ...
...                                     # add a colorbar
ax.set_xticks(range(4), cols, rotation=45, ha="right")
ax.set_yticks(range(4), cols)
plt.show()

In [ ]:
# Check
check_6_5()

<details><summary><b>Hint</b> (click to unfold)</summary>

`im = ax.imshow(corr, cmap="RdBu", vmin=-1, vmax=1)` draws the table; `fig.colorbar(im)` adds the colorbar.

</details>

**Example.** seaborn draws the same in one line, with the numbers in the cells:

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu", vmin=-1, vmax=1, ax=ax)
plt.show()

### From default to readable

**Example.** A default plot: no labels, no units, no message.

In [ ]:
fig, ax = plt.subplots()
ax.plot(df["social_media_hours"], df["grade"], "o")
plt.show()

**Your turn 6.6.** Make it readable: transparent dots, axis labels **with units**, and a title that **states the finding**. A reader should understand the plot without the surrounding text.

In [ ]:
fig, ax = plt.subplots()
...
plt.show()

In [ ]:
# Check
check_6_6()

### seaborn: the same in one line

seaborn works directly with a pandas table: name the columns, and `hue` colors the points by a third column, with a legend.

**Your turn 6.7.** Draw `study_hours` against `grade` with `sns.scatterplot`, colored by `passed`. Pass `ax=ax`, so that you can add labels afterwards.

In [ ]:
fig, ax = plt.subplots()
...
plt.show()

In [ ]:
# Check
check_6_7()

**Think.** Look at the colors. Where is the line between passed and failed, and why is it so sharp?

*Your answer:*

## Part 7. Checking data quality

Real survey exports are messy. `student_survey_raw.csv` is the same survey with the problems that real exports have. Lecture 2 gave four lines to run before any plot.

**Example.** Load the raw survey. Look closely at the `grade` column.

In [ ]:
raw = pd.read_csv(SURVEY_RAW)
raw.head()

**Your turn 7.1.** Run the four lines. Then store what you find: the total number of missing values in `n_missing`, the number of duplicate rows in `n_duplicates`, and the largest value of `sleep_hours` in `max_sleep`.

```python
raw.dtypes              # types per column
raw.isna().sum()        # missing values
raw.duplicated().sum()  # duplicate rows
raw.describe()          # min, max, mean, ...
```

In [ ]:
# Run the four lines here.

n_missing = ...
n_duplicates = ...
max_sleep = ...

In [ ]:
# Check
check_7_1()

**Explore: the detective.** `describe()` tells you that something is odd, not *who*. An interactive plot does. Run the cell, then **hover** over the points: the tooltip shows the row number (`index`) and the answers of that student.

**Your turn 7.2.** Find the student who is 210 years old and the one who sleeps 70 hours a night. Store their row numbers in `odd_age_row` and `odd_sleep_row`.

In [ ]:
import plotly.express as px

fig = px.scatter(raw.reset_index(), x="age", y="sleep_hours",
                 hover_data=["index", "program", "study_hours", "grade"],
                 opacity=0.6, width=650, height=400)
fig.show()

In [ ]:
odd_age_row = ...
odd_sleep_row = ...

In [ ]:
# Check
check_7_2()

Hovering is for exploring. Once you know what to look for, select with code, as in Part 2:

In [ ]:
raw[(raw["age"] > 100) | (raw["sleep_hours"] > 20)]

**Think.** What would you do with these two rows: delete them, repair them, or ask?

*Your answer:*

### Wrong data type: the grades

`raw.dtypes` says that `grade` is not a number: the grades were exported with a decimal comma, `"7,1"`. Text cannot be averaged or plotted on a numeric axis.

**Your turn 7.3.** Repair the column: replace the comma by a point, then convert to `float`.

In [ ]:
raw = pd.read_csv(SURVEY_RAW)       # a fresh copy, so that you can run this cell again
raw["grade"] = ...
raw["grade"].describe()

In [ ]:
# Check
check_7_3()

<details><summary><b>Hint</b> (click to unfold)</summary>

`raw["grade"].str.replace(",", ".")` replaces in every value of the column. Then add `.astype(float)`.

</details>

### Definition mismatch: the study hours

A problem that `describe()` does not show.

**Your turn 7.4.** Draw two histograms side by side, with 40 bins each: `study_hours` in `raw` on the left, `study_hours` in the clean `df` on the right.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
...
axes[0].set_title("raw survey")
axes[1].set_title("clean survey")
plt.show()

In [ ]:
# Check
check_7_4()

**Your turn 7.5.** On the left of the raw histogram there is a second peak. How many students report fewer than 4 study hours in `raw`, and how many in `df`? Store the counts in `n_low_raw` and `n_low_clean`.

In [ ]:
n_low_raw = ...
n_low_clean = ...

In [ ]:
# Check
check_7_5()

**Think.** What might these students have answered differently? Can you repair it without asking them?

*Your answer:*

### Missing values

**Your turn 7.6.** Which columns have gaps? Draw the missing values per column as horizontal bars, keeping only the columns that have missing values.

In [ ]:
missing = raw.isna().sum()
missing = ...                     # keep only the columns with missing values

fig, ax = plt.subplots(figsize=(5, 2))
...
ax.set_xlabel("missing values")
plt.show()

In [ ]:
# Check
check_7_6()

### A data-quality checklist

From Lecture 2. The last column says where the course returns to it.

| Question | Look at | Lecture |
|---|---|---|
| Right data types? | `dtypes` | 6 |
| Missing values? | `isna().sum()`, bar chart | 7 |
| Duplicate records? | `duplicated().sum()` | 6 |
| Very different scales? | `describe()` | 6 |
| Outliers and typing errors? | box plot, histogram | 13 |
| Same definition everywhere? | histogram | |
| Do you know what each column means? | the codebook | |

> **Garbage in, garbage out:** no model repairs data that are wrong.

## Part 8. Challenge: Dutch provinces

If time is left, or at home. Now a real dataset with less help; the checks still help you.

`cbs_provinces.csv` holds figures of Statistics Netherlands (CBS) for the 12 provinces, 2003 to 2017: one row per province and year. Among the columns: `total_population`, the population by age group (`nr_younger_than_5`, ..., `nr_between_65_and_80`, `nr_above_80`), `nr_in_urban_region`, `nr_buildings_per_km2` and `nr_personal_vehicles`.

**Your turn 8.1.** Check the data first. Store the number of provinces in `n_provinces` and the number of years in `n_years`. Are there missing values or duplicates?

In [ ]:
prov = pd.read_csv(PROVINCES)

n_provinces = ...
n_years = ...

In [ ]:
# Check
check_8_1()

**Your turn 8.2 (line plot).** Which province grows fastest? The provinces differ a lot in size, so plot the population **relative to 2003** (2003 = 100), one line per province. Store the name of the fastest-growing province in `fastest`.

In [ ]:
pop = prov.pivot(index="year", columns="province", values="total_population")
relative = ...

fig, ax = plt.subplots(figsize=(8, 4))
...

fastest = ...
plt.show()

In [ ]:
# Check
check_8_2()

<details><summary><b>Hint</b> (click to unfold)</summary>

`pop` has one column per province. `pop.loc[2003]` is the row of 2003; dividing `pop` by it divides each column by its own 2003 value. `relative.plot(ax=ax)` draws one line per column: the pandas shortcut from the table of libraries.

</details>

**Your turn 8.3 (bar chart).** Which province is the greyest? Compute the share of inhabitants aged 65 or older in 2017, and draw it as sorted horizontal bars. Store the greyest province in `greyest`.

In [ ]:
p2017 = prov[prov["year"] == 2017].set_index("province")
share_65plus = ...

fig, ax = plt.subplots()
...

greyest = ...
plt.show()

In [ ]:
# Check
check_8_3()

**Your turn 8.4 (definition check).** Plot `nr_in_urban_region` over time for Gelderland, Limburg and Noord-Brabant.

In [ ]:
urban = prov.pivot(index="year", columns="province", values="nr_in_urban_region")

fig, ax = plt.subplots()
...
plt.show()

**Think.** What happens in 2015? Is it likely that so many people moved in one year? What would you want to read before you use this column?

*Your answer:*

**Your turn 8.5.** Make one plot that shows something you did not know about the provinces, with a title that states the finding. Ideas: cars per 1000 inhabitants, the age structure of one province over time, buildings per km² against population.

In [ ]:
# Your plot here.

## Check your understanding

Exam-style questions, as in the lectures. Answer them without running code, then click **Check**.

In [ ]:
quiz_final()

## Where this is going

> Plot the data before modeling it. Plots reveal problems that summary numbers hide, and a few lines of `matplotlib` are enough.

**Coming up.** Lecture 3 (Monday, November 2): supervised learning, our first model ($k$-nearest neighbors), and how to test a model on new data. The survey stays with us: predicting `passed` from the other columns.

**Reading.** Müller and Guido (2016), Chapter 1: "Essential Libraries and Tools" and "First Things First: Look at Your Data".

**Further practice.** The [pandas getting-started tutorials](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html) and the [matplotlib quick start](https://matplotlib.org/stable/users/explain/quick_start.html).

---
*This lab follows Lecture 2 of Ş. İlker Birbil, which revises the slides of Stevan Rudinac and builds on Lecture 3 of Andreas Müller's Applied Machine Learning (Columbia University, 2017). Part 8 is adapted from the lab by Rens Dimmendaal used in earlier years of this course. Data: Anscombe (1973); math PhDs and arcade revenue as used by Müller (2017), after Vigen's Spurious Correlations; CBS StatLine. The student survey is synthetic.*